# Example: Interaction Analysis Species — Savannah Sparrow (*Passerculus sandwichensis*)

This notebook demonstrates the **Manuscript 1 interaction analysis pipeline** applied to Savannah Sparrow breeding season data. It serves as a worked example illustrating the full covariate selection sequence and GAM-based abundance modelling for a grassland bird species in relation to wind energy infrastructure and CRP grassland habitat.

**Species:** Savannah Sparrow (*Passerculus sandwichensis*)  
**Season:** Breeding (May–July)  
**Spatial grain:** 1.5 km hexagonal grid  
**Study period:** 2012–2021  
**Response variable:** Rounded mean eBird count per hexagon-year (`meanCount_round`)  
**Key finding:** Interaction model — `factor(WindHeight) * s(CRPGrass_area, k=5)` — selected as best

---

**Notebook structure:**
1. Setup & library loading
2. Data loading & joining
3. Prevalence & experimental group summaries
4. Wind categorical variable creation
5. Drop NAs & final prevalence
6. Correlation screening
7. Covariate distributions (EDA)
8. Scaling & train/test split
9. Count distribution & family selection
10. Effort variable selection
11. Land cover variable selection
12. Wind variable selection (simple spatial models)
13. Wind variable selection (full nuisance models)
14. CRP variable selection (simple spatial models)
15. CRP variable selection (full nuisance models)
16. Interaction vs additive model comparison
17. Best model summary & diagnostics
18. Prediction & visualization

---
*ecological_data_science - Samantha McGarrigle*

## 1. Setup

In [ ]:
library(dplyr)
library(tidyr)
library(ggplot2)
library(ggpubr)
library(mgcv)
library(zigam)
library(MuMIn)
library(corrplot)

# Resolve namespace conflicts
select <- dplyr::select
gam    <- mgcv::gam

# GAM smoothing parameters
k      <- 5  # basis dimension for most smooths
k_time <- 7  # basis dimension for cyclic time-of-day smooth

# Knot positions for meanStartTime cyclic spline
# Ensures the spline joins correctly at midnight even if no data fall near midnight
time_knots <- list(meanStartTime = seq(0, 24, length.out = k_time))

# Timing helper: wraps gam() and prints elapsed time
timed_gam <- function(formula, data, family = "nb", knots = time_knots) {
  t0 <- Sys.time()
  m  <- gam(formula, data = data, family = family, knots = knots)
  cat("Elapsed:", round(Sys.time() - t0, 2), "\n")
  m
}

# Wind binning helper: assigns "None" where no turbine present, otherwise cuts into bins
wind_bin <- function(x, pa, breaks, labels) {
  ifelse(pa == "N", "None",
         as.character(cut(x, breaks = breaks, labels = labels, right = TRUE)))
}

# EDA helper: density-overlaid histogram for a continuous variable
density_hist <- function(var, bw, base_size = 14) {
  ggplot(savspa_breed_1.5_NA, aes(x = .data[[var]])) +
    geom_histogram(aes(y = after_stat(density)), binwidth = bw,
                   color = "black", fill = "white") +
    geom_density(alpha = 0.2, fill = "blue") +
    labs(x = var) +
    theme_bw(base_size = base_size)
}

## 2. Data Loading & Joining

Three files are required:
- `species_season_finaldata.csv` — eBird hexagon-year data for Savannah Sparrow breeding season (152,322 rows x 45 cols)
- `WRD_foranalysis.csv` — Wind turbine attributes joined by `cell_year`
- CRP data file — joined by `seqnum` x `year`

This is an **interaction analysis**: both wind and CRP files are loaded.

In [ ]:
savspa_breed1_1.5  <- read.csv(file.choose())              # species_season_finaldata.csv
savspa_breed1b_1.5 <- savspa_breed1_1.5[, c(1:11, 18:45)] # retain effort, land cover, CRP, and response cols

In [ ]:
turbines <- read.csv(file.choose(), header = TRUE)  # WRD_foranalysis.csv
summary(turbines)

In [ ]:
# Join turbine attributes; fill WindCount = 0 where no turbines present
savspa_breed_1.5 <- left_join(savspa_breed1b_1.5, turbines, by = "cell_year") %>%
  mutate(WindCount = ifelse(is.na(WindCount), 0, WindCount))

summary(savspa_breed_1.5)

In [ ]:
CRP      <- read.csv(file.choose(), header = TRUE)  # CRP data file
CRP$year <- CRP$Year

# Join CRP metrics by seqnum x year (year-matched to account for annual CRP enrollment changes)
savspa_breed_1.5 <- savspa_breed_1.5 %>%
  left_join(select(CRP, seqnum, year, CRP_area, CRPGrass_area),
            by = c("seqnum" = "seqnum", "year" = "year"))

summary(savspa_breed_1.5)

## 3. Prevalence & Experimental Group Summaries

In [ ]:
# Detection prevalence
count(savspa_breed_1.5, species_observed) %>%
  mutate(percent = n / sum(n))

# Experimental group proportions: CRP x Wind presence combinations
groups <- list(
  "CRP + Wind" = savspa_breed_1.5$CRP_area > 0 & savspa_breed_1.5$WindCount > 0,
  "CRP only"   = savspa_breed_1.5$CRP_area > 0 & savspa_breed_1.5$WindCount == 0,
  "Wind only"  = savspa_breed_1.5$CRP_area == 0 & savspa_breed_1.5$WindCount > 0,
  "Neither"    = savspa_breed_1.5$CRP_area == 0 & savspa_breed_1.5$WindCount == 0
)
lapply(names(groups), function(g)
  cat(g, ": ", round(mean(groups[[g]]) * 100, 2), "%\n", sep = ""))
invisible(NULL)

## 4. Wind Categorical Variable Creation

Wind turbine attributes are recoded into ordered factor bins used as factors in GAMs.
All categories include a `"None"` level for hexagons with no turbines.

In [ ]:
savspa_breed_1.5 <- savspa_breed_1.5 %>%
  mutate(
    WindPA = factor(ifelse(WindCount > 0, "Y", "N")),

    WindAge_cat = factor(
      wind_bin(WindAge, WindPA,
               breaks = c(-Inf, 2, 4, 6, 8, Inf),
               labels = c("0-2", "3-4", "5-6", "7-8", "9+")),
      levels = c("None", "0-2", "3-4", "5-6", "7-8", "9+")),

    WindHeight_cat = factor(
      wind_bin(WindHeight, WindPA,
               breaks = c(0, 20, 40, 60, 80, Inf),
               labels = c("1-20", "21-40", "41-60", "61-80", "81-100+")),
      levels = c("None", "1-20", "21-40", "41-60", "61-80", "81-100+")),

    WindRSA_cat = factor(
      wind_bin(WindRSA, WindPA,
               breaks = c(0, 5000, 10000, 15000, Inf),
               labels = c("0-5000", "5001-10000", "10001-15000", "15000+")),
      levels = c("None", "0-5000", "5001-10000", "10001-15000", "15000+")),

    WindCap_cat = factor(
      wind_bin(WindCap, WindPA,
               breaks = c(0, 1000, 2000, 3000, 4000, Inf),
               labels = c("0-1000", "1001-2000", "2001-3000", "3001-4000", "4001+")),
      levels = c("None", "0-1000", "1001-2000", "2001-3000", "3001-4000", "4001+"))
  )

summary(select(savspa_breed_1.5, WindPA, WindAge_cat, WindHeight_cat, WindRSA_cat, WindCap_cat))

## 5. Drop NAs & Final Prevalence

Rows where wind attribute categories are `NA` arise from turbines with missing attribute data in the WRD. These are dropped prior to modelling.

In [ ]:
savspa_breed_1.5_NA <- savspa_breed_1.5 %>%
  drop_na(WindHeight_cat, WindCap_cat, WindRSA_cat)

str(savspa_breed_1.5_NA)

# Final detection prevalence and group counts after NA removal
count(savspa_breed_1.5_NA, species_observed) %>%
  mutate(percent = n / sum(n))

groups_NA <- list(
  "CRP + Wind" = savspa_breed_1.5_NA$CRP_area > 0 & savspa_breed_1.5_NA$WindCount > 0,
  "CRP only"   = savspa_breed_1.5_NA$CRP_area > 0 & savspa_breed_1.5_NA$WindCount == 0,
  "Wind only"  = savspa_breed_1.5_NA$CRP_area == 0 & savspa_breed_1.5_NA$WindCount > 0,
  "Neither"    = savspa_breed_1.5_NA$CRP_area == 0 & savspa_breed_1.5_NA$WindCount == 0
)
lapply(names(groups_NA), function(g)
  cat(g, ": ", round(mean(groups_NA[[g]]) * 100, 2), "%\n", sep = ""))
invisible(NULL)

## 6. Correlation Screening

Pairwise Pearson correlations are examined among land cover, effort, wind, and CRP covariates.
Threshold: |r| > 0.4.

**Key conflicts identified:**
- Cropland x TreeCover
- Cropland x CRP contagion
- GrassShrub x CRP/Grass area
- GrassShrub x CRP/Grass largest patch index

**Decision:** Remove GrassShrub and Cropland from the land cover set.

In [ ]:
corr_vars <- c(
  "WindCount",
  "Developed", "Cropland", "GrassShrub", "TreeCover", "Water", "Wetland", "Barren", "IceSnow",
  "meanStartTime", "meanSampDur", "meanDIST", "meanNumObs",
  "CRP_area", "CRP_largest_patch_index", "CRP_patch_density",
  "CRP_edge_density", "CRP_contagion",
  "CRPGrass_area", "CRPGrass_largest_patch_index", "CRPGrass_patch_density",
  "CRPGrass_edge_density", "CRPGrass_contagion",
  "Hab_PercentGrassland", "Hab_PercentWetland",
  "obj_PercentGrass", "obj_PercentWetland", "obj_PercentWildlife",
  "brd_PercentAttract", "brd_PercentNeutral", "brd_PercentAvoid"
)

corr_matrix <- cor(select(savspa_breed_1.5_NA, all_of(corr_vars)), use = "complete.obs")
write.csv(corr_matrix, file = "savspa_breed_1.5_corr_matrix.csv", row.names = TRUE)

options(repr.plot.width = 30, repr.plot.height = 30)
corrplot(corr_matrix, method = "circle", col = COL2("PiYG", 10),
         addCoef.col = "black", insig = "pch", pch = c("", "."),
         sig.level = 0.4)

## 7. Covariate Distributions (EDA)

Distributions of effort, land cover, wind, and CRP covariates are inspected prior to modelling.
Note: `density_hist()` is defined in Section 1 Setup.

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)

effort_plots <- list(
  density_hist("meanStartTime", bw = 2),
  density_hist("meanSampDur",   bw = 10),
  density_hist("meanDIST",      bw = 1),
  density_hist("meanNumObs",    bw = 1)
)
ggarrange(plotlist = effort_plots, ncol = 2, nrow = 2,
          labels = c("A", "B", "C", "D"))

In [ ]:
options(repr.plot.width = 30, repr.plot.height = 15)

lc_plots <- lapply(c("Developed", "Cropland", "GrassShrub", "TreeCover",
                     "Water", "Wetland", "Barren", "IceSnow"),
                   density_hist, bw = 10)
ggarrange(plotlist = lc_plots, ncol = 4, nrow = 2,
          labels = LETTERS[seq_along(lc_plots)])

In [ ]:
options(repr.plot.width = 30, repr.plot.height = 15)

wind_count_plot  <- density_hist("WindCount", bw = 25)
wind_bar_plots   <- lapply(
  c("WindHeight_cat", "WindCap_cat", "WindAge_cat", "WindPA", "WindRSA_cat"),
  function(v) ggplot(savspa_breed_1.5_NA, aes(x = .data[[v]])) +
    geom_bar() + labs(x = v) + theme_bw()
)
ggarrange(plotlist = c(list(wind_count_plot), wind_bar_plots),
          ncol = 3, nrow = 2, labels = LETTERS[1:6])

In [ ]:
options(repr.plot.width = 50, repr.plot.height = 20)

crp_pct_plots <- mapply(
  density_hist,
  var      = c("Hab_PercentGrassland", "Hab_PercentWetland", "obj_PercentGrass",
               "obj_PercentWetland", "obj_PercentWildlife", "brd_PercentAttract",
               "brd_PercentNeutral", "brd_PercentAvoid"),
  bw       = c(10, 10, 10, 10, 10, 10, 10, 10),
  base_size = 30,
  SIMPLIFY = FALSE
)
ggarrange(plotlist = crp_pct_plots, ncol = 4, nrow = 2,
          labels = LETTERS[seq_along(crp_pct_plots)])

In [ ]:
options(repr.plot.width = 50, repr.plot.height = 20)

crp_metric_specs <- list(
  c("CRP_area",                     250),
  c("CRP_largest_patch_index",      0.05),
  c("CRP_patch_density",            0.1),
  c("CRP_edge_density",             0.1),
  c("CRP_contagion",                0.1),
  c("CRPGrass_area",                250),
  c("CRPGrass_largest_patch_index", 0.05),
  c("CRPGrass_patch_density",       1),
  c("CRPGrass_edge_density",        10),
  c("CRPGrass_contagion",           0.1)
)
crp_metric_plots <- lapply(crp_metric_specs,
  function(s) density_hist(s[1], as.numeric(s[2]), base_size = 30))
ggarrange(plotlist = crp_metric_plots, ncol = 5, nrow = 2,
          labels = LETTERS[seq_along(crp_metric_plots)])

## 8. Scaling & Train/Test Split

All continuous predictors are standardized (mean = 0, SD = 1) prior to modelling.
Scaling parameters are saved for back-transformation of predictions.

In [ ]:
scale_vars <- c(
  "meanStartTime", "meanSampDur", "meanDIST", "meanNumObs",
  "Developed", "Cropland", "GrassShrub", "TreeCover",
  "Water", "Wetland", "Barren", "IceSnow",
  "WindCount",
  "Hab_PercentGrassland", "Hab_PercentWetland",
  "CRP_area", "CRP_largest_patch_index", "CRP_patch_density",
  "CRP_edge_density", "CRP_contagion",
  "CRPGrass_area", "CRPGrass_largest_patch_index", "CRPGrass_patch_density",
  "CRPGrass_edge_density", "CRPGrass_contagion",
  "obj_PercentGrass", "obj_PercentWetland", "obj_PercentWildlife",
  "brd_PercentAttract", "brd_PercentNeutral", "brd_PercentAvoid",
  "Lat", "Long"
)

# Save means and SDs for back-transformation
scaling_params <- savspa_breed_1.5_NA %>%
  select(all_of(scale_vars)) %>%
  summarise(across(everything(),
    list(mean = ~ mean(., na.rm = TRUE), sd = ~ sd(., na.rm = TRUE)))) %>%
  pivot_longer(everything(), names_to = c("variable", "statistic"),
               names_pattern = "(.*)_(mean|sd)") %>%
  pivot_wider(names_from = "statistic", values_from = "value")

scaling_params

In [ ]:
# Standardize, rename categorical wind cols, select modelling columns, split 80/20
savspa_breed_split <- savspa_breed_1.5_NA %>%
  mutate(across(all_of(scale_vars), scale)) %>%
  select(meanCount, meanCount_round,
         all_of(scale_vars),
         WindPA,
         WindHeight = WindHeight_cat,
         WindAge    = WindAge_cat,
         WindRSA    = WindRSA_cat,
         WindCap    = WindCap_cat) %>%
  split(if_else(runif(nrow(.)) <= 0.8, "train", "test"))

cat("Train:", nrow(savspa_breed_split$train),
    " Test:", nrow(savspa_breed_split$test), "\n")

## 9. Count Distribution & Family Selection

Four candidate families are compared using AIC on an intercept-only model: Negative Binomial, ZINB, Poisson, and Zero-Inflated Poisson.

**Result:** Negative Binomial selected (AIC ~81,400; substantially lower than all alternatives).

In [ ]:
f_intercept <- meanCount_round ~ 1

m_null_nb   <- timed_gam(f_intercept, savspa_breed_split$train, family = "nb")
m_null_p    <- timed_gam(f_intercept, savspa_breed_split$train, family = "poisson")
m_null_zip  <- timed_gam(f_intercept, savspa_breed_split$train, family = "ziP")
m_null_zinb <- zinbgam(f_intercept, pi.formula = ~ 1,
                       data = savspa_breed_split$train, knots = time_knots)

AIC(m_null_nb, m_null_p, m_null_zip)
cat("ZINB AIC:", m_null_zinb$aic, "\n")

## 10. Effort Variable Selection

Each effort covariate is tested individually against a spatial null (`s(Lat) + s(Long)`), then a global model with all four is compared using AICc.

**Result:** Global model selected — all four effort covariates retained.

In [ ]:
f_spatial <- meanCount ~ s(Lat, k=k) + s(Long, k=k)

f_eff_SampDur   <- update(f_spatial, ~ . + s(meanSampDur,   k=k))
f_eff_NumObs    <- update(f_spatial, ~ . + s(meanNumObs,    k=k))
f_eff_Dist      <- update(f_spatial, ~ . + s(meanDIST,      k=k))
f_eff_StartTime <- update(f_spatial, ~ . + s(meanStartTime, bs="cc", k=k_time))
f_eff_global    <- update(f_spatial, ~ . + s(meanSampDur, k=k) + s(meanNumObs, k=k) +
                            s(meanDIST, k=k) + s(meanStartTime, bs="cc", k=k_time))

m_eff_null      <- timed_gam(f_spatial,       savspa_breed_split$train)
m_eff_SampDur   <- timed_gam(f_eff_SampDur,   savspa_breed_split$train)
m_eff_NumObs    <- timed_gam(f_eff_NumObs,    savspa_breed_split$train)
m_eff_Dist      <- timed_gam(f_eff_Dist,      savspa_breed_split$train)
m_eff_StartTime <- timed_gam(f_eff_StartTime, savspa_breed_split$train)
m_eff_global    <- timed_gam(f_eff_global,    savspa_breed_split$train)

MuMIn::AICc(m_eff_null, m_eff_SampDur, m_eff_NumObs,
            m_eff_Dist, m_eff_StartTime, m_eff_global)

## 11. Land Cover Variable Selection

Each land cover covariate is tested individually against the spatial null. Collinear pairs with CRP variables (Cropland, GrassShrub — identified in Section 6) are excluded from the retained set.

**Result:** Retained — Developed, TreeCover, Water, Wetland, Barren, IceSnow.

In [ ]:
lc_candidate_vars <- c("Developed", "Cropland", "GrassShrub", "TreeCover",
                       "Water", "Wetland", "Barren", "IceSnow")

lc_formulas <- lapply(lc_candidate_vars, function(v)
  update(f_spatial, reformulate(c(".", paste0("s(", v, ", k=k)"))))
)
names(lc_formulas) <- lc_candidate_vars

m_lc_null  <- timed_gam(f_spatial, savspa_breed_split$train)
lc_models  <- lapply(lc_formulas, timed_gam, data = savspa_breed_split$train)

do.call(MuMIn::AICc, c(list(m_lc_null), lc_models))

## 12. Wind Variable Selection — Simple Spatial Models

Each wind variable is tested against a spatial null to assess marginal signal before adding nuisance covariates.

**Result:** WindHeight best in simple models.

In [ ]:
f_wind_simple <- list(
  WindCount  = update(f_spatial, ~ . + s(WindCount, k=k)),
  WindPA     = update(f_spatial, ~ . + factor(WindPA)),
  WindHeight = update(f_spatial, ~ . + factor(WindHeight)),
  WindAge    = update(f_spatial, ~ . + factor(WindAge)),
  WindRSA    = update(f_spatial, ~ . + factor(WindRSA)),
  WindCap    = update(f_spatial, ~ . + factor(WindCap))
)

m_wind_simple <- lapply(f_wind_simple, timed_gam, data = savspa_breed_split$train)

do.call(MuMIn::AICc, c(m_wind_simple, list(m_lc_null)))

## 13. Wind Variable Selection — Full Nuisance Models

Wind variables are evaluated with the full nuisance structure (effort + retained land cover + spatial).
Retained land cover: Developed, TreeCover, Water, Wetland, Barren, IceSnow.

**Result:** WindHeight best (AICc ~88,290), followed by WindRSA and WindCount. WindPA, WindAge, WindCap near null (AICc ~88,299).

In [ ]:
# Full nuisance base formula (effort + retained land cover + spatial)
# Note: IceSnow smoothed here based on univariate model performance; see pipeline notebook
f_nuisance <- meanCount ~ s(meanSampDur, k=k) + s(meanNumObs, k=k) + s(meanDIST, k=k) +
  s(meanStartTime, bs="cc", k=k_time) +
  s(Developed, k=k) + s(TreeCover, k=k) + s(Water, k=k) +
  s(Wetland, k=k) + s(Barren, k=k) + s(IceSnow, k=k) +
  s(Lat, k=k) + s(Long, k=k)

f_wind_nuisance <- list(
  WindCount  = update(f_nuisance, ~ . + WindCount),
  WindPA     = update(f_nuisance, ~ . + factor(WindPA)),
  WindHeight = update(f_nuisance, ~ . + factor(WindHeight)),
  WindAge    = update(f_nuisance, ~ . + factor(WindAge)),
  WindRSA    = update(f_nuisance, ~ . + factor(WindRSA)),
  WindCap    = update(f_nuisance, ~ . + factor(WindCap))
)

m_wind_nuisance <- lapply(f_wind_nuisance, timed_gam, data = savspa_breed_split$train)
m_wind_null     <- timed_gam(f_nuisance, savspa_breed_split$train)

do.call(MuMIn::AICc, c(m_wind_nuisance, list(m_wind_null)))

## 14. CRP Variable Selection — Simple Spatial Models

Each CRP covariate is tested individually against a spatial null. Variables with near-zero or bounded distributions are entered as linear terms; the remainder as smooths.

**Result:** CRPGrass_contagion best in simple models (AICc ~93,814).

In [ ]:
crp_smooth_vars <- c(
  "CRP_area", "CRP_largest_patch_index", "CRP_patch_density",
  "CRP_edge_density", "CRP_contagion",
  "CRPGrass_area", "CRPGrass_largest_patch_index", "CRPGrass_patch_density",
  "CRPGrass_edge_density", "CRPGrass_contagion",
  "Hab_PercentGrassland", "obj_PercentGrass",
  "brd_PercentAttract", "brd_PercentAvoid"
)
crp_linear_vars <- c("Hab_PercentWetland", "obj_PercentWetland",
                     "obj_PercentWildlife", "brd_PercentNeutral")

f_crp_simple_smooth <- setNames(
  lapply(crp_smooth_vars, function(v)
    update(f_spatial, reformulate(c(".", paste0("s(", v, ", k=k)"))))),
  crp_smooth_vars)

f_crp_simple_linear <- setNames(
  lapply(crp_linear_vars, function(v) update(f_spatial, reformulate(c(".", v)))),
  crp_linear_vars)

m_crp_simple_smooth <- lapply(f_crp_simple_smooth, timed_gam, data = savspa_breed_split$train)
m_crp_simple_linear <- lapply(f_crp_simple_linear, timed_gam, data = savspa_breed_split$train)
m_crp_simple_null   <- timed_gam(f_spatial, savspa_breed_split$train)

do.call(MuMIn::AICc,
        c(m_crp_simple_smooth, m_crp_simple_linear, list(m_crp_simple_null)))

## 15. CRP Variable Selection — Full Nuisance Models

CRP variables are evaluated with the full nuisance structure.

**Result:** CRPGrass_area best (AICc ~87,805), followed by CRPGrass_largest_patch_index (~87,938) and CRPGrass_contagion (~87,989).

In [ ]:
f_crp_smooth <- setNames(
  lapply(crp_smooth_vars, function(v)
    update(f_nuisance, reformulate(c(".", paste0("s(", v, ", k=k)"))))),
  crp_smooth_vars)

f_crp_linear <- setNames(
  lapply(crp_linear_vars, function(v) update(f_nuisance, reformulate(c(".", v)))),
  crp_linear_vars)

m_crp_smooth <- lapply(f_crp_smooth, timed_gam, data = savspa_breed_split$train)
m_crp_linear <- lapply(f_crp_linear, timed_gam, data = savspa_breed_split$train)
m_crp_null   <- timed_gam(f_nuisance, savspa_breed_split$train)

do.call(MuMIn::AICc, c(m_crp_smooth, m_crp_linear, list(m_crp_null)))

## 16. Interaction vs Additive Model Comparison

The two best single-variable models (WindHeight, CRPGrass_area) are combined into additive and interaction models.

| Model | AICc |
|-------|------|
| Additive: `factor(WindHeight) + s(CRPGrass_area)` | **~87,795** |
| Interaction: `factor(WindHeight) * CRPGrass_area` | ~87,798 |
| CRPGrass_area alone | ~87,805 |
| WindHeight alone | ~88,290 |

**Decision:** Additive model selected — interaction term provides no meaningful improvement in AICc.

In [ ]:
f_wind_height_final <- update(f_nuisance, ~ . + factor(WindHeight))
f_crp_CGArea        <- update(f_nuisance, ~ . + s(CRPGrass_area, k=k))
f_Height_CGArea_add <- update(f_nuisance, ~ . + factor(WindHeight) + s(CRPGrass_area, k=k))
f_Height_CGArea_int <- update(f_nuisance, ~ . + factor(WindHeight) + s(CRPGrass_area, k=k) +
                                factor(WindHeight):CRPGrass_area)

m_wind_height_final <- timed_gam(f_wind_height_final, savspa_breed_split$train)
m_crp_CGArea        <- timed_gam(f_crp_CGArea,        savspa_breed_split$train)
m_Height_CGArea_add <- timed_gam(f_Height_CGArea_add, savspa_breed_split$train)
m_Height_CGArea_int <- timed_gam(f_Height_CGArea_int, savspa_breed_split$train)

MuMIn::AICc(m_wind_height_final, m_crp_CGArea,
            m_Height_CGArea_add, m_Height_CGArea_int)

## 17. Best Model Summary & Diagnostics

Best model: `m_Height_CGArea_add` — `factor(WindHeight) + s(CRPGrass_area, k=5)` + full nuisance structure.

In [ ]:
summary(m_Height_CGArea_add)

In [ ]:
gam.vcomp(m_Height_CGArea_add, rescale = TRUE)

In [ ]:
anova(m_Height_CGArea_add)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 20)
plot(m_Height_CGArea_add)

## 18. Prediction & Visualization

Predictions are generated on the test set and CRPGrass_area is back-transformed to its original scale.
Predicted abundance is plotted against CRP grassland area, coloured by wind turbine height category.

In [ ]:
crp_mean <- scaling_params$mean[scaling_params$variable == "CRPGrass_area"]
crp_sd   <- scaling_params$sd[scaling_params$variable   == "CRPGrass_area"]

pred_df <- savspa_breed_split$test %>%
  mutate(
    predicted              = predict.gam(m_Height_CGArea_add,
                               newdata = savspa_breed_split$test,
                               type    = "response"),
    CRPGrass_area_original = CRPGrass_area * crp_sd + crp_mean
  )

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 7)

ggplot(pred_df, aes(x = CRPGrass_area_original, y = predicted, color = WindHeight)) +
  geom_smooth() +
  labs(
    x     = "CRP Grassland Area (ha)",
    y     = "Predicted Abundance",
    color = "Wind Turbine Height",
    title = "Savannah Sparrow: Predicted Abundance vs. CRP Grassland Area by Wind Turbine Height"
  ) +
  theme_bw(base_size = 13)